In [1]:
# importing the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import os
import joblib
from sklearn.preprocessing import MinMaxScaler

In [2]:
# loading the datasets

telco = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
subscription = pd.read_csv("../data/raw/Subscription_Service_Churn_Dataset.csv")
ecom = pd.read_csv("../data/raw/ecommerce_transactions.csv")

In [3]:
# Standardizing the column names

telco.columns = telco.columns.str.lower().str.strip()
subscription.columns = subscription.columns.str.lower().str.strip()
ecom.columns = ecom.columns.str.lower().str.strip()

In [4]:
# Columns present in subscription table

print(subscription.columns.tolist())

['customer_id', 'age', 'gender', 'subscription_length', 'region', 'payment_method', 'support_tickets_raised', 'satisfaction_score', 'discount_offered', 'last_activity', 'monthly_spend', 'churned']


In [5]:
# Columns present in telco table

print(telco.columns.tolist())

['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


In [6]:
# Columns present in ecom table

print(ecom.columns.tolist())

['transaction_id', 'user_name', 'age', 'country', 'product_category', 'purchase_amount', 'payment_method', 'transaction_date']


# Deciding columns we will be keeping inside the merged dataset (for now)
customer_id
gender
age
tenure
monthlycharges
balance
yearly_salary
paymentmethod
churn
industry

In [7]:
# Editing column names to ease the process of merging

telco.rename(columns={'customerid' : 'customer_id'}, inplace=True)

#subscription.rename(columns={'customerid' : 'customer_id'}, inplace=True)
subscription.rename(columns={'payment_method' : 'paymentmethod'}, inplace=True)
subscription.rename(columns={'subscription_length' : 'tenure'}, inplace=True)
subscription.rename(columns={'monthly_spend' : 'monthlycharges'}, inplace=True)
subscription.rename(columns={'churned' : 'churn'}, inplace=True)

ecom.rename(columns={'payment_method' : 'paymentmethod'}, inplace=True)

In [8]:
# Creating customer_id column in the e-commerce dataset

# Adding prefix to make customer_IDs unique
telco['customer_id'] = telco['customer_id'].apply(lambda x: f"TEL_{x}")
#subscription['customer_id'] = subscription['customer_id'].apply(lambda x: f"SUBS_{x}")

# Creating customer_id in e-commerce dataset
if 'user_name' in ecom.columns:
    user_id_map = {name: f"ECOM_{idx+1}" for idx, name in enumerate(ecom['user_name'].unique())}
    ecom['customer_id'] = ecom['user_name'].map(user_id_map)

# Creating customer_id in subscription dataset
if 'customer_id' in subscription.columns:
    user_id_map = {name: f"SUBS_{idx+1}" for idx, name in enumerate(subscription['customer_id'].unique())}
    subscription['customer_id'] = subscription['customer_id'].map(user_id_map)


In [9]:
# checking if missing values are present

#df_subs_7043.info()

-------------------------------------------------------------------------------------------------------------

In [10]:
#creating churn column in ecommerce dataset

mask_ecom = ecom['customer_id'].str.startswith('ECOM_')
options = [0, 1]
ecom.loc[mask_ecom, 'churn'] = np.random.choice(options, size=mask_ecom.sum())

In [11]:
#undersampling ecommerce dataset using stratification

def stratified_undersample(df, target_size, stratify_cols, random_state=42):
    df = df.copy()
    
    # Create stratification key
    df["_strat_key"] = df[stratify_cols].astype(str).agg("_".join, axis=1)

    total = len(df)
    frac = target_size / total

    parts = []

    # Stratified proportional sampling
    for key, group in df.groupby("_strat_key"):
        n = max(1, int(len(group) * frac))
        parts.append(group.sample(n=n, random_state=random_state))

    df_partial = pd.concat(parts, ignore_index=True)

    # If partial sample is smaller than target → sample extra rows
    if len(df_partial) < target_size:
        needed = target_size - len(df_partial)
        extra = df.sample(n=needed, random_state=random_state)
        df_partial = pd.concat([df_partial, extra], ignore_index=True)

    # If partial sample is larger → trim
    df_final = df_partial.sample(n=target_size, random_state=random_state)

    return df_final.drop(columns=["_strat_key"]).reset_index(drop=True)

target_size = 7043
strat_cols = ["churn", "paymentmethod"]
df_ecom_7k = stratified_undersample(ecom, target_size, strat_cols)

print(df_ecom_7k.shape)
print(df_ecom_7k["churn"].value_counts())
print(df_ecom_7k["paymentmethod"].value_counts())
#print(df_ecom_7k["Gender"].value_counts())

(7043, 10)
churn
0.0    3522
1.0    3521
Name: count, dtype: int64
paymentmethod
UPI                 1193
Cash on Delivery    1188
Debit Card          1178
Credit Card         1170
PayPal              1163
Net Banking         1151
Name: count, dtype: int64


In [12]:
#oversampling subscription dataset

def oversample_to_target(df, target_size, random_state=42, noise_std=1e-6):
    
    df = df.copy().reset_index(drop=True)
    current_size = len(df)

    # If already big enough → simply downsample
    if current_size >= target_size:
        return df.sample(n=target_size, random_state=random_state).reset_index(drop=True)

    needed = target_size - current_size

    # Identify numeric columns for noise addition
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Oversample using bootstrapping
    df_extra = df.sample(n=needed, replace=True, random_state=random_state).copy()

    # Add tiny noise ONLY to numeric columns to avoid duplicates
    for col in numeric_cols:
        df_extra[col] += np.random.normal(0, noise_std, size=len(df_extra))

    # Combine original + new oversampled rows
    df_final = pd.concat([df, df_extra], ignore_index=True)

    return df_final.reset_index(drop=True)


target_size = 7043  # your required size
df_subs_7043 = oversample_to_target(subscription, target_size)

print(df_subs_7043.shape)

(7043, 12)


---------------------------------------------------------------------------------------------------

In [13]:
# Combining the datasets

#finding all the columns in the dataset
all_cols = set(telco.columns) | set(df_subs_7043.columns) | set(df_ecom_7k.columns)

#adding missing columns to the datasets
for df in [telco, df_subs_7043, df_ecom_7k]:
    for col in all_cols:
        if col not in df.columns:
            df[col] = None

#Reordering the columns in a similar order in all datasets
telco = telco[sorted(all_cols)]
df_subs_7043 = df_subs_7043[sorted(all_cols)]
df_ecom_7k = df_ecom_7k[sorted(all_cols)]

#Combining the datasets (row-wise)
combined_df = pd.concat([telco, df_subs_7043, df_ecom_7k], ignore_index=True)

/var/folders/vz/dx6hxp0d2dggzmjsyf6_pzv00000gn/T/ipykernel_77192/663097727.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([telco, df_subs_7043, df_ecom_7k], ignore_index=True)


In [14]:
# checking if missing values are present

combined_df_backup.info()

NameError: name 'combined_df_backup' is not defined

In [ ]:
# Saving intermediate cleaned dataset

combined_df.to_csv("encoded_churn_dataset_2.csv", index=False)
print("Dataset saved successfully")

In [15]:
#filling in the dataset with gender

def assign_gender(name):
    name = str(name).split()[0].lower()
    if name.endswith(('a', 'e', 'ie')):
        return 'Female'
    else:
        return 'Male'

ecom_mask = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'gender'] = combined_df.loc[ecom_mask,'user_name'].apply(assign_gender)

In [16]:
# filling in the tenure column for customer_ids starting with 'ECOM_'

ecom_mask = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'tenure'] = np.random.randint(1, 70, size=ecom_mask.sum())

#sub_mask = combined_df['customer_id'].str.startswith('SUBS_')
#combined_df.loc[sub_mask, 'tenure'] = np.random.randint(1, 70, size=sub_mask.sum())

In [17]:
# filling in random monthlycharges value for customers with id 'ECOM_'

mask_ecom = combined_df['customer_id'].str.startswith('ECOM_')
combined_df.loc[ecom_mask, 'monthlycharges'] = np.round(np.random.uniform(20, 120, size=mask_ecom.sum()), 2)

#mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
#missing_subs = mask_subs & combined_df['monthlycharges'].isna()
#combined_df.loc[missing_subs, 'monthlycharges'] = np.round(np.random.uniform(20, 120, size=missing_subs.sum()), 2)

In [ ]:
#editing the payment methods column

#payment_options = ['Electronic check', 'Mailed check', 'PayPal', 'Debit Card', 'Credit Card', 'Bank transfer']
#missing = combined_df['paymentmethod'].isna()
#combined_df.loc[missing, 'paymentmethod'] = np.random.choice(payment_options, size=missing.sum())

In [18]:
#editing the churn column

#ecom doesn't have churn so randomly assigning the values
#mask_ecom = combined_df['customer_id'].str.startswith('ECOM_')
#options = ['Yes', 'No']
#combined_df.loc[mask_ecom, 'churn'] = np.random.choice(options, size=mask_ecom.sum())

#mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
#combined_df.loc[mask_subs, 'churn'] = combined_df.loc[mask_subs, 'churn'].replace({1: 'Yes', 0: 'No'})


#converting churn values of customer_id 'SUB_S' to a proper value
mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
combined_df.loc[mask_subs, 'churn'] = (combined_df.loc[mask_subs, 'churn'].astype(float) > 0.5).astype(int)

#converting yes to 1 and no to 0 for churn column for customer's whose id starts with 'TEL_'
mask_tel = combined_df['customer_id'].str.startswith('TEL_')
combined_df.loc[mask_tel, 'churn'] = (combined_df.loc[mask_tel, 'churn'].replace({'Yes' : 1, 'No' : 0, 'yes' : 1, 'no' : 0}).astype(int))


/var/folders/vz/dx6hxp0d2dggzmjsyf6_pzv00000gn/T/ipykernel_77192/23771064.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined_df.loc[mask_tel, 'churn'] = (combined_df.loc[mask_tel, 'churn'].replace({'Yes' : 1, 'No' : 0, 'yes' : 1, 'no' : 0}).astype(int))


In [19]:
#editing tenure column to remove decimal values
mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
combined_df.loc[mask_subs, 'tenure'] = combined_df.loc[mask_subs, 'tenure'].round().astype(int)

In [20]:
#converting the datatypes of age and tenure to int

combined_df['age'] = combined_df['age'].astype(int)
combined_df['tenure'] = combined_df['tenure'].astype(int)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [21]:
# making the industry column

conditions = [
    combined_df['customer_id'].str.startswith('TEL_'),
    combined_df['customer_id'].str.startswith('SUBS_'),
    combined_df['customer_id'].str.startswith('ECOM_')]

values = ['Telecom', 'Subscription', 'Ecommerce']

combined_df['industry'] = np.select(conditions, values, default = 'Unknown')

In [22]:
# copying the accountage values to tenure column for customer_id's starting 'SUBS_'

mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
combined_df.loc[mask_subs, 'tenure'] = combined_df.loc[mask_subs, 'accountage']

KeyError: 'accountage'

In [ ]:
# creating a complete backup before dropping any columns

combined_df_backup = combined_df.copy()

In [ ]:
#Dropping unnecessary columns

columns_to_keep = [
    'customer_id',
    'gender',
    'age',
    'tenure',
    'monthlycharges',
    'paymentmethod',
    'churn',
    'industry']

combined_df = combined_df[columns_to_keep]

In [23]:
#assigning ages randomly to the rows where age is missing
# filling in the age column for customer_ids starting with 'TEL_'

mask_tel = combined_df['customer_id'].str.startswith('TEL_')
combined_df.loc[mask_tel, 'age'] = np.random.randint(20, 71, size=mask_tel.sum())

mask_subs = combined_df['customer_id'].str.startswith('SUBS_')
mask_missing_age = mask_subs & combined_df['age'].isna()
combined_df.loc[mask_subs, 'age'] = np.random.randint(20, 71, size=mask_subs.sum())

In [ ]:
combined_df = combined_df_backup.copy()

In [ ]:
# checking if missing values are present

combined_df.info()

In [ ]:
--------------------------------------------------------------------------------------------------

In [26]:
# Encoding categorical features (i.e. converting text to numerical by assigning them numbers)

encoder = LabelEncoder()
for col in ['gender', 'paymentmethod', 'industry']:
    combined_df[col] = encoder.fit_transform(combined_df[col])
    

#df_encoded = combined_df.copy()

#Label Encoding (gender)
#encoder = LabelEncoder()
#for col in ['gender']:
#df_encoded['gender'] = encoder.fit_transform(df_encoded['gender'])

#ONE-HOT Encoding
#one_hot_cols = ['paymentmethod', 'industry']
#df_encoded = pd.get_dummies(df_encoded, columns=one_hot_cols, drop_first=True)

#checking results
#print(df_encoded.head())
#print(df_encoded.shape)

[0 1]
[0 1 2 3 4 5 6 7 8 9]
[0 1 2]
[0 1 2]


In [27]:
#taking a backup of combined_df after encoding

combined_df_encoded = combined_df.copy()

In [42]:
#downloading combined_df and combined_df_backup datasets into data folder

save_path = "../data"

os.makedirs(save_path, exist_ok=True)

combined_df.to_csv(os.path.join(save_path, "combined_cleaned_encoded.csv"), index=False)
combined_df_backup.to_csv(os.path.join(save_path, "combined_cleaned_unencoded.csv"), index=False)

---------------------------------------------------------------------------------------------------

In [ ]:
---------------------------------------------------------------------------------------------------

In [58]:
#filling in the dataset with gender

#def assign_gender(name):
 #   name = str(name).split()[0].lower()
  #  if name.endswith(('a', 'e', 'ie')):
   #     return 'Female'
    #else:
     #   return 'Male'

#subs_mask = combined_df_backup['customer_id'].str.startswith('SUBS_')
#combined_df_backup.loc[subs_mask, 'gender'] = combined_df_backup.loc[subs_mask,'user_name'].apply(assign_gender)

In [66]:
# defining set (X) and target (Y) columns

X = combined_df[['gender', 'age', 'tenure', 'monthlycharges', 'paymentmethod', 'industry']]
Y = combined_df['churn']

In [77]:
# Encoding categorical features in Y dataset

encoder_y = LabelEncoder()
Y = encoder_y.fit_transform(Y)
Y_df = pd.DataFrame(Y, columns=['churn'])

In [79]:
#spliting before applying SMOTE

X_train, X_test, Y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [81]:
#applying SMOTE only on training set

smote = SMOTE(random_state=42)
X_train_bal, Y_train_bal = smote.fit_resample(X_train, Y_train)

Y_train_series = pd.Series(Y_train)
Y_train_bal_series = pd.Series(Y_train_bal)

print("Before SMOTE: ", Y_train_series.value_counts().to_dict())
print("After SMOTE: ", Y_train_bal_series.value_counts().to_dict())
print(f"Training data shape changed from {X_train.shape} --> {X_train_bal.shape}")

Before SMOTE:  {0: 24670, 1: 21734}
After SMOTE:  {1: 24670, 0: 24670}
Training data shape changed from (46404, 6) --> (49340, 6)


what the output above means??
BEFORE SMOTE: {0: 24670, 1: 21734}
0 --> non-churn customers
1 --> churned customers
we had few churners than non-churners (imbalanced data)

AFTER SMOTE: {1: 24670, 0: 24670}
SMOTE added new synthetic churn (1) samples (making classes equal, helping model learn about both groups equally)

CHANGE IN DATASET: (46404, 6) --> (49340, 6)
Before SMOTE: 46404 rows
After SMOTE: 49340 rows


In [87]:
#downloading combined_df and combined_df_backup datasets into data folder

save_path = "../data"

os.makedirs(save_path, exist_ok=True)

combined_df.to_csv(os.path.join(save_path, "combined_cleaned.csv"), index=False)
combined_df_backup.to_csv(os.path.join(save_path, "combined_cleaned_initial.csv"), index=False)

In [90]:
#downloading train and test data files

save_path = "../data"
os.makedirs(save_path, exist_ok=True)

X_train_df = pd.DataFrame(X_train_bal, columns=X_train.columns)
Y_train_df = pd.DataFrame(Y_train_bal, columns=['churn'])
X_test_df = pd.DataFrame(X_test, columns=X_test.columns)
y_test_df = pd.DataFrame(y_test, columns=['churn'])

try:
    X_train_df.to_csv(os.path.join(save_path, "X_train_bal.csv"), index=False)
    Y_train_df.to_csv(os.path.join(save_path, "Y_train_bal.csv"), index=False)
    X_test_df.to_csv(os.path.join(save_path, "X_test.csv"), index=False)
    y_test_df.to_csv(os.path.join(save_path, "y_test.csv"), index=False)
    print("SMOTE based train/test sets saved")
except Exception as e:
    print("Could not save", e)

SMOTE based train/test sets saved


In [94]:
# saving encoder

try:
    joblib.dump(encoder_y, os.path.join(save_path, "label_encoder.pkl"))
    print("Encoder saved")
except Exception as e:
    print("Encoder not found or not saved")

Encoder saved


#scalers aren't used in the code for now since we don't need to normalize or standardize the numeric columns for now
# but if we need it in the future then we can do and save it as follows
# scaler helps bring features like age, tenure and monthlycharges to similar numerical scales thus
# improving performance for distance-based or gradient-based models like Logistic Regression, KNN,
# SVM, Neural Nets
# if we use models like Random Forest, XGBoost, then we don't need scalar

try:
    joblib.dump(scaler, os.path.join(save_path, "scaler.pkl"))
    print("✅ Scaler saved.")
except Exception as e:
    print("⚠️ Scaler not found or not saved:", e)

#to load all the files back and run the codes and models further, do the following first always

import pandas as pd
import joblib

#load datasets
combined_df = pd.read_csv("../data/combined_cleaned.csv")
combined_df_backup = pd.read_csv("../data/combined_cleaned_initial.csv")

#load SMOTE data (if needed)
X_train_bal = pd.read_csv("../data/X_train_bal.csv")
Y_train_bal = pd.read_csv("../data/Y_train_bal.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_test = pd.read_csv("../data/y_test.csv")

# loading encoders (or scalers if we use in the future)
encoder = joblib.load("..data/label_encoder.pkl")
scaler = joblib.load("..data/scaler.pkl")